# 06 - Soil State Transformer Architecture

Design and prototype the Soil State Transformer (SST) foundation model.

## Architecture Overview
```
Soil State Transformer (SST)
├── Static Context Encoder
│   ├── Soil properties → Linear(n_soil, 64)
│   ├── Topography → Linear(n_topo, 64)
│   └── Location → Fourier → Linear(4, 32)
│
├── Temporal Sequence Encoder
│   ├── Weather sequence → Linear(n_weather, 64)
│   ├── Temporal position → Sinusoidal encoding
│   └── Transformer blocks (6 layers, 8 heads)
│
└── Prediction Heads
    ├── Nutrient prediction → Linear(hidden, n_targets)
    └── Uncertainty → Linear(hidden, n_targets) → Softplus
```

In [ ]:
# Standard imports
import sys
sys.path.insert(0, '../..')

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Configuration
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

RESULTS_DIR = Path('../../results/model_experiments')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## 1. Model Configuration

In [ ]:
# Hyperparameters to prototype
class SSTConfig:
    """Configuration for Soil State Transformer."""
    
    # Input dimensions
    n_static_features: int = 20      # Soil + topography + location
    n_weather_features: int = 8      # ERA5 variables per timestep
    max_seq_length: int = 365        # Days of weather history
    n_targets: int = 6               # SOC, pH, clay, sand, N, CEC
    
    # Model dimensions
    embedding_dim: int = 256         # Main hidden dimension
    static_embed_dim: int = 128      # Static context embedding
    ff_dim: int = 1024               # Feed-forward dimension
    
    # Transformer
    n_heads: int = 8
    n_layers: int = 6
    dropout: float = 0.1
    
    # Training
    mask_ratio: float = 0.15         # For pre-training

config = SSTConfig()
print("Model configuration:")
for key, value in vars(config).items():
    if not key.startswith('_'):
        print(f"  {key}: {value}")

## 2. Positional Encoding

In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    """
    Sinusoidal positional encoding for temporal sequences.
    
    Encodes position (day) as combination of sin/cos waves at
    different frequencies, allowing model to learn relative positions.
    """
    
    def __init__(self, d_model: int, max_len: int = 365):
        super().__init__()
        
        # Create positional encoding matrix
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        # Register as buffer (not a parameter)
        self.register_buffer('pe', pe.unsqueeze(0))
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Add positional encoding to input."""
        return x + self.pe[:, :x.size(1)]

# Visualize positional encoding
pe = SinusoidalPositionalEncoding(d_model=128, max_len=365)

fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(pe.pe[0].numpy().T, aspect='auto', cmap='RdBu')
ax.set_xlabel('Position (day)')
ax.set_ylabel('Dimension')
ax.set_title('Sinusoidal Positional Encoding')
plt.colorbar(im)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'positional_encoding.png', dpi=150)
plt.show()

## 3. Static Context Encoder

In [ ]:
class StaticContextEncoder(nn.Module):
    """
    Encode static features (soil, topography, location).
    
    Combines:
    - Soil properties (SoilGrids or local measurements)
    - Topographic features (elevation, slope, TWI, etc.)
    - Location (Fourier-encoded lat/lon)
    """
    
    def __init__(
        self,
        n_soil_features: int = 6,
        n_topo_features: int = 10,
        embed_dim: int = 128,
        dropout: float = 0.1
    ):
        super().__init__()
        
        # Soil encoder
        self.soil_encoder = nn.Sequential(
            nn.Linear(n_soil_features, 64),
            nn.LayerNorm(64),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        # Topography encoder
        self.topo_encoder = nn.Sequential(
            nn.Linear(n_topo_features, 64),
            nn.LayerNorm(64),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        # Location encoder (Fourier features)
        self.location_encoder = nn.Sequential(
            nn.Linear(4, 32),  # sin/cos of lat and lon
            nn.ReLU()
        )
        
        # Combine all static features
        self.combiner = nn.Sequential(
            nn.Linear(64 + 64 + 32, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
    
    def encode_location(self, lat: torch.Tensor, lon: torch.Tensor) -> torch.Tensor:
        """Encode lat/lon as Fourier features."""
        # Normalize to [-1, 1]
        lat_norm = lat / 90.0
        lon_norm = lon / 180.0
        
        # Fourier features
        features = torch.stack([
            torch.sin(math.pi * lat_norm),
            torch.cos(math.pi * lat_norm),
            torch.sin(math.pi * lon_norm),
            torch.cos(math.pi * lon_norm)
        ], dim=-1)
        
        return features
    
    def forward(
        self,
        soil_features: torch.Tensor,
        topo_features: torch.Tensor,
        lat: torch.Tensor,
        lon: torch.Tensor
    ) -> torch.Tensor:
        """
        Encode static context.
        
        Returns:
            Static context embedding [batch, embed_dim]
        """
        soil_embed = self.soil_encoder(soil_features)
        topo_embed = self.topo_encoder(topo_features)
        
        loc_features = self.encode_location(lat, lon)
        loc_embed = self.location_encoder(loc_features)
        
        combined = torch.cat([soil_embed, topo_embed, loc_embed], dim=-1)
        return self.combiner(combined)

# Test static encoder
static_encoder = StaticContextEncoder()
batch_size = 4

soil = torch.randn(batch_size, 6)
topo = torch.randn(batch_size, 10)
lat = torch.randn(batch_size) * 30 + 40  # ~40°N ± 30
lon = torch.randn(batch_size) * 30 - 95  # ~95°W ± 30

static_embed = static_encoder(soil, topo, lat, lon)
print(f"Static context shape: {static_embed.shape}")

## 4. Temporal Sequence Encoder

In [ ]:
class TemporalSequenceEncoder(nn.Module):
    """
    Transformer encoder for weather time series.
    
    Processes daily weather data with:
    - Linear projection to embedding dimension
    - Sinusoidal positional encoding
    - Multi-layer transformer encoder
    """
    
    def __init__(
        self,
        n_weather_features: int = 8,
        embed_dim: int = 256,
        n_heads: int = 8,
        n_layers: int = 6,
        ff_dim: int = 1024,
        dropout: float = 0.1,
        max_len: int = 365
    ):
        super().__init__()
        
        # Input projection
        self.input_proj = nn.Linear(n_weather_features, embed_dim)
        
        # Positional encoding
        self.pos_encoder = SinusoidalPositionalEncoding(embed_dim, max_len)
        
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=n_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        
        self.norm = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)
    
    def forward(
        self,
        weather_seq: torch.Tensor,
        mask: torch.Tensor = None
    ) -> torch.Tensor:
        """
        Encode weather sequence.
        
        Args:
            weather_seq: [batch, seq_len, n_features]
            mask: Optional attention mask
        
        Returns:
            Encoded sequence [batch, seq_len, embed_dim]
        """
        # Project to embedding dimension
        x = self.input_proj(weather_seq)
        
        # Add positional encoding
        x = self.pos_encoder(x)
        x = self.dropout(x)
        
        # Transformer encoding
        x = self.transformer(x, src_key_padding_mask=mask)
        x = self.norm(x)
        
        return x

# Test temporal encoder
temporal_encoder = TemporalSequenceEncoder(
    n_weather_features=8,
    embed_dim=256,
    n_heads=8,
    n_layers=6
)

weather_seq = torch.randn(batch_size, 365, 8)
temporal_embed = temporal_encoder(weather_seq)
print(f"Temporal sequence shape: {temporal_embed.shape}")

## 5. Complete Soil State Transformer

In [ ]:
class SoilStateTransformer(nn.Module):
    """
    Soil Dynamics Foundation Model.
    
    Combines static context and temporal weather sequences
    to predict soil nutrient properties.
    
    Pre-training: Masked property prediction
    Fine-tuning: LoRA adaptation for local fields
    """
    
    def __init__(self, config: SSTConfig):
        super().__init__()
        self.config = config
        
        # Static context encoder
        self.static_encoder = StaticContextEncoder(
            n_soil_features=6,
            n_topo_features=10,
            embed_dim=config.static_embed_dim,
            dropout=config.dropout
        )
        
        # Temporal sequence encoder
        self.temporal_encoder = TemporalSequenceEncoder(
            n_weather_features=config.n_weather_features,
            embed_dim=config.embedding_dim,
            n_heads=config.n_heads,
            n_layers=config.n_layers,
            ff_dim=config.ff_dim,
            dropout=config.dropout,
            max_len=config.max_seq_length
        )
        
        # Cross-attention: temporal attends to static
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=config.embedding_dim,
            num_heads=config.n_heads,
            dropout=config.dropout,
            batch_first=True
        )
        
        # Project static to temporal dimension
        self.static_proj = nn.Linear(config.static_embed_dim, config.embedding_dim)
        
        # Aggregation: pool temporal sequence
        self.temporal_pool = nn.Sequential(
            nn.Linear(config.embedding_dim, config.embedding_dim),
            nn.Tanh()
        )
        self.pool_weights = nn.Linear(config.embedding_dim, 1)
        
        # Prediction heads
        self.predictor = nn.Sequential(
            nn.Linear(config.embedding_dim, config.ff_dim),
            nn.LayerNorm(config.ff_dim),
            nn.ReLU(),
            nn.Dropout(config.dropout),
            nn.Linear(config.ff_dim, config.n_targets)
        )
        
        # Uncertainty head (log variance)
        self.uncertainty = nn.Sequential(
            nn.Linear(config.embedding_dim, config.ff_dim // 2),
            nn.ReLU(),
            nn.Linear(config.ff_dim // 2, config.n_targets),
            nn.Softplus()  # Ensure positive variance
        )
    
    def attention_pool(self, x: torch.Tensor) -> torch.Tensor:
        """Attention-based pooling over temporal dimension."""
        # x: [batch, seq, embed]
        weights = self.pool_weights(self.temporal_pool(x))  # [batch, seq, 1]
        weights = F.softmax(weights, dim=1)
        pooled = (weights * x).sum(dim=1)  # [batch, embed]
        return pooled
    
    def forward(
        self,
        soil_features: torch.Tensor,
        topo_features: torch.Tensor,
        lat: torch.Tensor,
        lon: torch.Tensor,
        weather_seq: torch.Tensor
    ) -> tuple:
        """
        Forward pass.
        
        Returns:
            predictions: [batch, n_targets]
            uncertainty: [batch, n_targets]
        """
        # Encode static context
        static = self.static_encoder(soil_features, topo_features, lat, lon)
        static = self.static_proj(static).unsqueeze(1)  # [batch, 1, embed]
        
        # Encode temporal sequence
        temporal = self.temporal_encoder(weather_seq)  # [batch, seq, embed]
        
        # Cross-attention: temporal attends to static
        attended, _ = self.cross_attention(temporal, static, static)
        temporal = temporal + attended
        
        # Pool temporal representation
        pooled = self.attention_pool(temporal)  # [batch, embed]
        
        # Predictions
        predictions = self.predictor(pooled)
        uncertainty = self.uncertainty(pooled)
        
        return predictions, uncertainty

# Test complete model
model = SoilStateTransformer(config)

# Count parameters
n_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {n_params:,}")
print(f"  Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# Forward pass
predictions, uncertainty = model(soil, topo, lat, lon, weather_seq)
print(f"\nPredictions shape: {predictions.shape}")
print(f"Uncertainty shape: {uncertainty.shape}")

## 6. Loss Functions

In [ ]:
class GaussianNLLLoss(nn.Module):
    """
    Gaussian Negative Log-Likelihood loss with uncertainty.
    
    Allows model to learn both prediction and uncertainty.
    """
    
    def forward(
        self,
        predictions: torch.Tensor,
        targets: torch.Tensor,
        variance: torch.Tensor
    ) -> torch.Tensor:
        """Calculate NLL loss."""
        # variance = σ²
        loss = 0.5 * (
            torch.log(variance) + 
            (targets - predictions) ** 2 / variance
        )
        return loss.mean()


class MaskedPropertyLoss(nn.Module):
    """
    Loss for masked property prediction (pre-training).
    
    Only computes loss on masked positions.
    """
    
    def __init__(self):
        super().__init__()
        self.mse = nn.MSELoss(reduction='none')
    
    def forward(
        self,
        predictions: torch.Tensor,
        targets: torch.Tensor,
        mask: torch.Tensor
    ) -> torch.Tensor:
        """Calculate loss only on masked positions."""
        loss = self.mse(predictions, targets)
        masked_loss = loss * mask
        return masked_loss.sum() / (mask.sum() + 1e-8)

# Test losses
nll_loss = GaussianNLLLoss()
masked_loss = MaskedPropertyLoss()

targets = torch.randn(batch_size, 6)
mask = torch.rand(batch_size, 6) > 0.85

print(f"NLL Loss: {nll_loss(predictions, targets, uncertainty):.4f}")
print(f"Masked Loss: {masked_loss(predictions, targets, mask.float()):.4f}")

## 7. Model Summary

In [ ]:
# Print model architecture
print("Soil State Transformer Architecture:")
print("=" * 60)
print(model)

# Parameter breakdown
print("\n" + "=" * 60)
print("Parameter Breakdown:")
print("-" * 60)

for name, module in model.named_children():
    n_params = sum(p.numel() for p in module.parameters())
    print(f"{name:30} {n_params:>12,}")

print("-" * 60)
print(f"{'TOTAL':30} {sum(p.numel() for p in model.parameters()):>12,}")

In [ ]:
# Save model architecture code
import json

architecture_summary = {
    'model_name': 'SoilStateTransformer',
    'total_parameters': sum(p.numel() for p in model.parameters()),
    'config': vars(config),
    'components': {
        'static_encoder': sum(p.numel() for p in model.static_encoder.parameters()),
        'temporal_encoder': sum(p.numel() for p in model.temporal_encoder.parameters()),
        'cross_attention': sum(p.numel() for p in model.cross_attention.parameters()),
        'predictor': sum(p.numel() for p in model.predictor.parameters()),
        'uncertainty': sum(p.numel() for p in model.uncertainty.parameters())
    }
}

with open(RESULTS_DIR / 'sst_architecture.json', 'w') as f:
    json.dump(architecture_summary, f, indent=2)

print(f"Architecture summary saved to {RESULTS_DIR / 'sst_architecture.json'}")

## 8. Next Steps

This notebook defined the SST architecture. Next:

1. **07_temporal_encoding.ipynb**: Explore temporal encoding variations
2. **08_lora_finetuning.ipynb**: Implement LoRA for efficient fine-tuning
3. **Pre-training**: Train on WoSIS global data
4. **Evaluation**: Compare with baseline models